<a href="https://colab.research.google.com/github/Najaf-Ali12/Complete-NLP-Projects/blob/main/Customer_Query_Categorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importing necessary libraries
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
import wandb  # for curve visualization


In [2]:
# Loading dataset
dataset=load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset",split="train")
data=dataset.to_pandas()

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [3]:
data.columns
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [4]:
# Checking for duplicates and null values
data.duplicated().sum()
data.isnull().sum()

,0
flags,0
instruction,0
category,0
intent,0
response,0


In [5]:
# Removing unwanted columns
data=data.drop(["flags","intent","response"],axis=1)
data.columns

Index(['instruction', 'category'], dtype='object')

In [6]:
# Checking for data imbalancing
data['category'].value_counts()

,count
category,
ACCOUNT,5986
ORDER,3988
REFUND,2992
CONTACT,1999
INVOICE,1999
PAYMENT,1998
FEEDBACK,1997
DELIVERY,1994
SHIPPING,1970


In [7]:
# Handling imbalanced data
from sklearn.utils import resample
import pandas as pd

# Set the target number of rows you want for EVERY single class
TARGET_SAMPLE_COUNT = 2500

balanced_df_list = []

# Loop through each individual class partition inside your category column
for category_name, group in data.groupby('category'):
    if len(group) < TARGET_SAMPLE_COUNT:
        # UPSAMPLE: If the class has fewer rows than target, sample with replacement
        group_resampled = resample(group,
                                   replace=True,
                                   n_samples=TARGET_SAMPLE_COUNT,
                                   random_state=42)
    else:
        # DOWNSAMPLE: If the class has more rows than target, sample without replacement
        group_resampled = resample(group,
                                   replace=False,
                                   n_samples=TARGET_SAMPLE_COUNT,
                                   random_state=42)

    balanced_df_list.append(group_resampled)

# Combine all perfectly balanced blocks back into one data frame
balanced_data = pd.concat(balanced_df_list).sample(frac=1, random_state=42).reset_index(drop=True)

print("--- Balanced Class Distribution (Pandas) ---")
print(balanced_data['category'].value_counts())


--- Balanced Class Distribution (Pandas) ---
category
SHIPPING        2500
INVOICE         2500
CANCEL          2500
DELIVERY        2500
PAYMENT         2500
REFUND          2500
CONTACT         2500
FEEDBACK        2500
SUBSCRIPTION    2500
ORDER           2500
ACCOUNT         2500
Name: count, dtype: int64


In [8]:
# Creating mapping between the labels and their names
unique_categories = sorted(balanced_data['category'].unique().tolist())
label2id={label:id for id,label in enumerate(unique_categories)}
id2label={id:label for label,id in label2id.items()}


In [9]:
# Lowering the text as it is liked by distilbert
data['instruction']=data['instruction'].str.lower()


In [10]:
# Tokenization
from datasets import Dataset
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
  return tokenizer(examples['instruction'],padding=True, Truncation=True, max_length=512)

# Converting pandas dataframe into hugging face dataset to eliminate the confusion of .map() as it is both in pandas and hf
dataset=Dataset.from_pandas(data)

# Now use hf .map() method that applies tokenize_function on each data sample of dataset
tokenized_dataset=dataset.map(tokenize_function,batched=True)
tokenized_dataset

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Dataset({
    features: ['instruction', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 26872
})

In [11]:
# Splitting dataset into training testing and validation and its structuring

# First 80% for training and 20% for testing
train_test_split=tokenized_dataset.train_test_split(test_size=0.2)

# Further 20% is divided into 50% testing and 50% in validation
test_val_split=train_test_split["test"].train_test_split(test_size=0.5)

# Now combining all three training, testing and validation dataset into clean hugging face dataset structure
from datasets import DatasetDict
dataset=DatasetDict({
    "train":train_test_split["train"],
    "validation":test_val_split["train"],  # first half(50%) of test_val_split is kept for validation
    "test":test_val_split["test"]          # Second half(50%) of test_val_split is kept for testing.
})

dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21497
    })
    validation: Dataset({
        features: ['instruction', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2687
    })
    test: Dataset({
        features: ['instruction', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2688
    })
})

In [12]:
# Converting Category column in numerical form and
# renaming it as label because hugging face models require target column named as "label"
def convert_category_to_id(example):
    # Get the text category string (e.g., 'ACCOUNT')
    category_string = example['category']

    # Look up the corresponding number ID from your dictionary
    numeric_id = label2id[category_string]

    # Return a dictionary containing the new 'label' key Hugging Face requires
    return {'label': numeric_id}

dataset=dataset.map(convert_category_to_id)

# Removing unnecessary category_column
dataset=dataset.remove_columns("category")

dataset


Map:   0%|          | 0/21497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 21497
    })
    validation: Dataset({
        features: ['instruction', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 2687
    })
    test: Dataset({
        features: ['instruction', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 2688
    })
})

In [13]:
# Loading the pre-trained model for sequence classification
model=AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=10,
    id2label=id2label, # with these two parameters we ensure that model output our labels instead of dummy label_0,label_1 etc
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
# Providing write access token to login in hugging face and push training progress directly to hugging face
from huggingface_hub import login
from google.colab import userdata

# using secret
write_token=userdata.get("hf_write_token")

# Paste your copied HF write token inside the quotes
login(write_token)

In [23]:
# Setting training argumens
arguments=TrainingArguments(
    output_dir="./QueryCategorizer",  # where to save output files like checkpoints, configuration jsons and weights
    eval_strategy="epoch",  # eval model after end of each epoch
    save_strategy="epoch",  # save model performance snapshots/checkpoints after end of each epoch. It is from where you can resume.
    learning_rate=0.00002,
    per_device_train_batch_size=16,     # Feeds 16 text strings into the GPU compute architecture simultaneously per batch.
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,  #  Regularization penality
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=True,
    warmup_steps=500,
    eval_steps=500,
    save_steps=500,
    logging_steps=200,
    report_to="wandb"
)



In [24]:
# Writing function of metrics for evaluation
def compute_metrics(eval_pred):
  predictions,label=eval_pred
  predictions=np.argmax(predictions,axis=1)

  accuracy=accuracy_score(predictions,label)
  precision=precision_score(predictions,label,average="weighted")
  recall=recall_score(predictions,label,average="weighted")
  f1=f1_score(predictions,label,average="weighted")

  return {
      "accuracy":accuracy,
      "precision":precision,
      "recall":recall,
      "f1":f1
  }


In [25]:
# Initializing the training
trainer=Trainer(
    model=model,
    args=arguments,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # if after 3 epochs f1_score not improves stop training.
)
# because in Training Arguments we set metrix_for_best_model="f1" and greater_is_better=True.

In [27]:
# Initialize weights and biases for experiement tracking and visualizing the model performance during its training
wandb.init(project="transformer-fine-tuning", name="bert-mrpc-analysis")

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [28]:
# Training
trainer.train()

# Here iam facing issue of data leakage because i performed upsampling in data that lead to duplicates and that's why my data is getting leaked.

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.006073,0.002772,0.999256,0.999259,0.999256,0.999256
2,0.003421,0.000272,1.000000,1.000000,1.000000,1.000000
3,0.001495,0.000112,1.000000,1.000000,1.000000,1.000000
4,0.000119,0.000044,1.000000,1.000000,1.000000,1.000000
5,0.000050,0.000022,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Evaluating model on validation data

print("-"*30)
print("Model Evaluation on Validation set")
print("-"*30)

eval_results=trainer.evaluate(dataset["validation"])

print("*"*20," Validation Results ","*"*20)
for matrix,result in eval_results:
  print(f"{matrix}:{result}")